# 📝 Smart Campus Lost & Found — NLP Text Similarity Matching

This notebook demonstrates the **Natural Language Processing (NLP)** pipeline for discovering lost items when a user does not have a photograph.

### Objectives:
1. **Information Asymmetry Solution:** Enable students to search campus records using natural language descriptions.
2. **Statistical Model:** Fit a Scikit-Learn TF-IDF vectorizer (unigrams + bigrams) on the 120,000 campus descriptions.
3. **Deep Learning Model:** Train a custom PyTorch Continuous Bag-of-Words (CBOW) Word2Vec network from scratch on campus text.
4. **Hybrid Retrieval:** Ensemble keyword overlap with dense semantic embeddings for ultra-fast (<6ms) cosine similarity matching.

In [ ]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np

# Add ml-service root to Python path
ML_SERVICE_DIR = os.path.abspath(os.path.join("..", "ml-service"))
if ML_SERVICE_DIR not in sys.path:
    sys.path.insert(0, ML_SERVICE_DIR)

from text_matching.preprocessor import clean_text, tokenize_text
from text_matching.tfidf_model import CampusTFIDFMatcher
from text_matching.word2vec_model import CampusWord2Vec
from text_matching.matcher import TextMatcher

print("[OK] Dependencies successfully imported.")

## 1. Load Campus Lost & Found Descriptions (120k Dataset)

In [ ]:
DATA_PATH = os.path.join("..", "data", "smart_campus_dataset_120k.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records.")
print("\nSample Campus Descriptions:")
for d in df["description"].dropna().head(5):
    print(f"  • {d}")

## 2. Text Preprocessing & Tokenization

In [ ]:
raw_sample = "Blue Decathlon Water Bottle, left on a table at Central Library"
print("Raw Text:       ", raw_sample)
print("Cleaned Text:   ", clean_text(raw_sample, remove_stopwords=False))
print("Tokens (Filtered):", tokenize_text(raw_sample))

## 3. Load Trained NLP Models (TF-IDF & PyTorch Word2Vec)

In [ ]:
MODELS_DIR = os.path.join(ML_SERVICE_DIR, "models")
tfidf = CampusTFIDFMatcher()
tfidf.load(os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib"))

w2v = CampusWord2Vec()
w2v.load(os.path.join(MODELS_DIR, "word2vec"))

catalog_data = joblib.load(os.path.join(MODELS_DIR, "found_text_catalog.joblib"))
catalog_tfidf = catalog_data["tfidf_matrix"]
catalog_w2v = catalog_data["w2v_matrix"]
metadata = catalog_data["metadata"]
print(f"[OK] Loaded models and {len(metadata):,} indexed found item descriptions.")

## 4. Run Semantic Similarity Retrieval (Lost vs. Found Query)

In [ ]:
matcher = TextMatcher(tfidf_matcher=tfidf, word2vec_model=w2v)

query = "Blue water bottle lost near Central Library"
print(f"Query: \"{query}\"\n")

t0 = time.time()
matches = matcher.match(
    query_text=query,
    catalog_tfidf_matrix=catalog_tfidf,
    catalog_w2v_matrix=catalog_w2v,
    catalog_metadata=metadata,
    top_k=5,
    filter_category="Sports Equipment"
)
print(f"Searched in {(time.time() - t0)*1000:.2f}ms:\n")
for m in matches:
    print(f"#{m['rank']}: [{m['confidence_percent']}% Match] {m['item_name']} at {m['location']}")
    print(f"     Description: \"{m['description']}\"")